In [1]:
# Clone the official UR-DMU repository
# This repository is used as the base framework for inference
# and evaluation on the XD-Violence dataset.
!git clone https://github.com/henrryzh1/UR-DMU.git

# Move into the repository directory
%cd UR-DMU

Cloning into 'UR-DMU'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 66 (delta 12), reused 8 (delta 8), pack-reused 41 (from 1)
Receiving objects: 100% (66/66), 104.12 MiB | 41.88 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/kaggle/working/UR-DMU


In [2]:
# List repository contents to verify correct cloning
!ls /kaggle/working/UR-DMU

AD_Vis.py	   frame_label	models	    translayer.py  xd_infer.py
config.py	   LICENSE	options.py  ucf_infer.py   xd_main.py
data		   list		outputs     ucf_main.py    xd_test.py
dataset_loader.py  memory.py	README.md   ucf_test.py
feature_extract    model.py	train.py    utils.py


In [3]:
# ============================================================
# Dataset list adaptation (environment-dependent)
# ------------------------------------------------------------
# The XD_Test.list file is environment-specific, as it only
# contains absolute paths to pre-extracted feature files.
#
# For this reason, the original list provided by the UR-DMU
# repository is replaced with a locally adapted version,
# where paths are updated to match the Kaggle file system.
#
# This file depends on the local
# storage structure. Only feature paths are modified:
# - video ordering is preserved
# - labels and annotations are unchanged
# - no methodological changes are introduced
# ============================================================
!cp /kaggle/input/hl-net/list_URDMU/list_URDMU/XD_Test.list /kaggle/working/UR-DMU/list/XD_Test.list

In [4]:
# Install additional dependencies required by the UR-DMU framework
!pip install visdom
!pip install ipdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for visdom: filename=visdom-0.2.4-py3-none-any.whl size=1408195 sha256=3ef45b0e5d4efbd3e4f410af6a1b6e747ea615b2d92395672dddc7ae9abc741f
  Stored in directory: /root/.cache/pip/wheels/fa/a4/bb/2be445c295d88a74f9c0a4232f04860ca489a5c7c57eb959d9
Successfully built visdom


In [5]:
%%writefile /kaggle/working/UR-DMU/xd_infer.py
import torch
import numpy as np
from dataset_loader import XDVideo
from options import parse_args
import pdb
from config import Config
import utils
import os
from model import WSAD
from tqdm import tqdm
from dataset_loader import data
from sklearn.metrics import roc_curve,auc,precision_recall_curve
def valid(net, config, test_loader, model_file = None):
    """
    Perform inference on the test set and compute frame-level predictions.

    This function saves frame-level anomaly scores and computes
    both video-level and frame-level evaluation metrics.
    """
    with torch.no_grad():
        net.eval()
        net.flag = "Test"
        if model_file is not None:
            net.load_state_dict(torch.load(model_file))
            
        pre_dict = {}
        gt_dict = {}
        load_iter = iter(test_loader)
        frame_gt = np.load("frame_label/xd_gt.npy")
        frame_predict = None
        cls_label = []
        cls_pre = []
        count = 0
        for i in tqdm(range(len(test_loader.dataset)//5)):

            _data, _label = next(load_iter)
            
            _data = _data.cuda()
            _label = _label.cuda()

            cls_label.append(int(_label[0]))
            res = net(_data)   
            a_predict = res["frame"].cpu().numpy().mean(0)   
            cls_pre.append(1 if a_predict.max()>0.5 else 0)          
            fpre_ = np.repeat(a_predict,16)
            pl = len(fpre_)
            pre_dict[i] = fpre_
            gt_dict[i] = frame_gt[count: count+pl]
            count = count + pl
            if frame_predict is None:         
                frame_predict = fpre_
            else:
                frame_predict = np.concatenate([frame_predict, fpre_])   
        
        # Save frame-level predictions and auxiliary dictionaries
        np.save('frame_label/xd_frame_pre.npy', frame_predict)
        np.save('frame_label/xd_pre_dict.npy', pre_dict)
        np.save('frame_label/xd_gt_dict.npy', gt_dict)
        fpr,tpr,_ = roc_curve(frame_gt, frame_predict)
        auc_score = auc(fpr, tpr)
        print("auc:{}".format(auc_score))
        corrent_num = np.sum(np.array(cls_label) == np.array(cls_pre), axis=0)
        accuracy = corrent_num / (len(cls_pre))
        precision, recall, th = precision_recall_curve(frame_gt, frame_predict,)
        ap_score = auc(recall, precision)

        print("accuracy:{}".format(accuracy))
        print("ap_score:{}".format(ap_score))
        # Subset-based evaluation (only anomalous frames)
        anomaly_mask = np.load('/kaggle/input/hl-net/xd_anomaly_mask_BNWVAD.npy')
        sub_predict = frame_predict[anomaly_mask]
        sub_gt = frame_gt[anomaly_mask]
    
        fpr,tpr,_ = roc_curve(sub_gt, sub_predict)
        auc_sub = auc(fpr, tpr)

        precision, recall, th = precision_recall_curve(sub_gt, sub_predict)
        ap_sub = auc(recall, precision)
        print("auc_sub:{}".format(auc_sub))
        print("ap_sub:{}".format(ap_sub))
         
if __name__ == "__main__":
    args = parse_args()
    if args.debug:
        pdb.set_trace()
    config = Config(args)
    worker_init_fn = None
    config.len_feature = 1024
    if config.seed >= 0:
        utils.set_seed(config.seed)
        worker_init_fn = np.random.seed(config.seed)
    net = WSAD(config.len_feature, flag = "Test", a_nums = 60, n_nums = 60)
    net = net.cuda()
    test_loader = data.DataLoader(
        XDVideo(root_dir = config.root_dir, mode = 'Test', modal = config.modal, num_segments = config.num_segments, len_feature = config.len_feature),
            batch_size = 5,
            shuffle = False, num_workers = config.num_workers,
            worker_init_fn = worker_init_fn)
    valid(net, config, test_loader, model_file = os.path.join(args.model_path, "xd_trans_2022.pkl"))

Overwriting /kaggle/working/UR-DMU/xd_infer.py


In [6]:
# Run the custom inference script
!python /kaggle/working/UR-DMU/xd_infer.py 

100%|█████████████████████████████████████████| 800/800 [01:14<00:00, 10.79it/s]
auc:0.9401785378050785
accuracy:0.9
ap_score:0.8166405599487506
auc_sub:0.823644933010847
ap_sub:0.8285477372881345


In [10]:
def load_video_boundaries(list_file):
    """
    Compute the number of clips for each video listed in a .list file.
    """
    paths = [l.strip() for l in open(list_file)]
    video_info = {}
    for p in paths: 
        fname = p.split("/")[-1] 
        base = "__".join(fname.split("__")[:-1]) 
        if base not in video_info:
            video_info[base] = 0
        video_info[base] += 1 
    return list(video_info.items()) 


def detect_fight_videos(list_file):
    """
    Identify fight videos based on label information encoded
    in the file names (label 'B1').
    """
    paths = [l.strip() for l in open(list_file)]
    fight_dict = {}
    for p in paths:
        fname = p.split("/")[-1]
        base = "__".join(fname.split("__")[:-1])
        if "_label_" in base: 
            after = base.split("_label_")[1]
            labels = after.split("-")
            is_fight = any(lbl == "B1" for lbl in labels)
            fight_dict[base] = int(is_fight)
        else:
            fight_dict[base] = 0
    return fight_dict 


In [11]:
def get_fight_frames(pred, gt, list_file):
    video_info = load_video_boundaries(list_file)
    fight_dict = detect_fight_videos(list_file)

    frame_idx = 0
    fight_pred = []
    fight_gt = []

    for video_name, num_clips in video_info:
        frames = num_clips * 16
        if fight_dict[video_name] == 1: 
            fight_pred.extend(pred[frame_idx: frame_idx + frames])
            fight_gt.extend(gt[frame_idx: frame_idx + frames])
        else:
            fight_pred.extend([0] * frames)
            fight_gt.extend([0] * frames)
        frame_idx += frames

    return np.array(fight_pred), np.array(fight_gt)

In [12]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc as sk_auc

# Load frame-level predictions and ground truth
pred = np.load("/kaggle/working/UR-DMU/frame_label/xd_frame_pre.npy")  
gt = np.load("/kaggle/working/UR-DMU/frame_label/xd_gt.npy")           
fight_pred, fight_gt = get_fight_frames(pred, gt, "/kaggle/working/UR-DMU/list/XD_Test.list")

# Precision-Recall curve and F1-score
precision, recall, thresholds = precision_recall_curve(fight_gt, fight_pred)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)

best_idx = f1_scores.argmax()
best_f1 = f1_scores[best_idx]
best_precision = precision[best_idx]
best_recall = recall[best_idx]

print("Best F1:", best_f1)
print(" Best Precision:", best_precision)
print(" Best Recall:", best_recall)

# PR-AUC and ROC-AUC
pr_auc = sk_auc(recall, precision)
print("PR-AUC:", pr_auc)
roc_auc = roc_auc_score(fight_gt, fight_pred)
print("ROC-AUC:", roc_auc)


Best F1: 0.8058300395251928
 Best Precision: 0.770308564231738
 Best Recall: 0.84478591160221
PR-AUC: 0.8789886259090871
ROC-AUC: 0.9876458476972654


In [13]:
def get_fight_videos_for_sub(pred, gt, list_file):
    """
    Extract frame-level predictions and ground truth for subset-based
    evaluation, considering only fight videos with at least one
    positive frame.
    """
    video_info = load_video_boundaries(list_file)
    fight_dict = detect_fight_videos(list_file)

    frame_idx = 0

    sub_pred = []
    sub_gt = []

    for video_name, num_clips in video_info:
        frames = num_clips * 16

        if fight_dict[video_name] == 1:

            vid_pred = pred[frame_idx: frame_idx + frames]
            vid_gt   = gt[frame_idx: frame_idx + frames]

            # Ensure at least one positive frame
            if np.sum(vid_gt) > 0:
                sub_pred.extend(vid_pred)
                sub_gt.extend(vid_gt)

        frame_idx += frames

    return np.array(sub_pred), np.array(sub_gt)

In [14]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve
fight_pred_sub, fight_gt_sub = get_fight_videos_for_sub(
    pred, gt, "/kaggle/working/UR-DMU/list/XD_Test.list"
)

fpr,tpr,_ = roc_curve(fight_gt_sub, fight_pred_sub)
auc_metric = auc(fpr, tpr)
print("ROC-AUC_sub (fight videos only):", auc_metric)

precision_sub, recall_sub, _ = precision_recall_curve(fight_gt_sub, fight_pred_sub)
pr_auc_sub = auc(recall_sub, precision_sub)
print("PR-AUC_sub (fight videos only)::", pr_auc_sub)


ROC-AUC_sub (fight videos only): 0.6604260186464089
PR-AUC_sub (fight videos only):: 0.9389013134286168
